In [1]:
#TO DO : adding the markdown titles, ajouter les cellules Camembert et LGBM si elles fonctionnent, afficher à chaque nouvel ajout de features les résultats en accuracy et en loss du xgboost
#interpréter quelles features sont les plus significatives pour distinguer observateur d'influenceur

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, accuracy_score
import numpy as np
#!pip install xgboost
import xgboost as xgb
import json
from pandas import json_normalize
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
#!pip install textblob
from textblob import TextBlob
import nltk
nltk.download('stopwords')
import re
from gensim.models import Word2Vec
from gensim.models import Word2Vec

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


ModuleNotFoundError: No module named 'gensim'

In [ ]:
# Loading the training data from a JSON Lines file (one JSON object per line)
def load_jsonl_skip_bad(path):
    data_list = []
    with open(path, "r") as f:
        for line in f:
            try:
                data_list.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return json_normalize(data_list)

train_data = pd.read_json('train.jsonl',lines="True")
train_data = json_normalize(train_data.to_dict(orient='records'))

kaggle_data = pd.read_json('kaggle_test.jsonl',lines="True")
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

In [ ]:
def create_advanced_features(df_input):
    df = df_input.copy()

    default_int_series = pd.Series(0, index=df.index)
    default_bool_series = pd.Series(False, index=df.index)

    #basic user and tweet metrics
    df['user.followers_count'] = df.get('user.followers_count', default_int_series).fillna(0)
    df['user.friends_count'] = df.get('user.friends_count', default_int_series).fillna(0)
    df['user.listed_count'] = df.get('user.listed_count', default_int_series).fillna(0)
    df['user.favourites_count'] = df.get('user.favourites_count', default_int_series).fillna(0)
    df['user.statuses_count'] = df.get('user.statuses_count', default_int_series).fillna(0)
    df['retweet_count'] = df.get('retweet_count', default_int_series).fillna(0)
    df['favorite_count'] = df.get('favorite_count', default_int_series).fillna(0)
    df['quote_count'] = df.get('quote_count', default_int_series).fillna(0)
    df['reply_count'] = df.get('reply_count', default_int_series).fillna(0)

    #account creation date
    df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
    ref_date = pd.to_datetime('now', utc=True)
    df['account_age_days'] = (ref_date - df['user_created_at_dt']).dt.days
    df['account_age_days'] = df['account_age_days'].fillna(0)

    #additional time features
    df['created_at_dt'] = pd.to_datetime(df.get('created_at'), errors='coerce')
    df['tweet_hour'] = df['created_at_dt'].dt.hour.fillna(-1)
    df['tweet_is_weekend'] = df['created_at_dt'].dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    #profile quality
    df['is_default_profile'] = df.get('user.default_profile', default_bool_series).fillna(False).astype(int)
    df['is_default_image'] = df.get('user.default_profile_image', default_bool_series).fillna(False).astype(int)
    df['is_verified'] = df.get('user.verified', default_bool_series).fillna(False).astype(int)
    df['is_protected'] = df.get('user.protected', default_bool_series).fillna(False).astype(int)
    df['has_url'] = df.get('user.url', pd.Series(False, index=df.index)).notna().astype(int)

    #tweet content entity counting
    def count_entities(x):
        if isinstance(x, list) or (isinstance(x, pd.Series) and x.dtype == object): return len(x)
        return 0

    df['num_urls'] = df.get('entities.urls', default_int_series).apply(count_entities)
    df['num_hashtags'] = df.get('entities.hashtags', default_int_series).apply(count_entities)
    df['num_mentions'] = df.get('entities.user_mentions', default_int_series).apply(count_entities)
    df['has_media'] = df.get('extended_entities.media', default_bool_series).notna().astype(int)

    #ratio features
    followers = df['user.followers_count']
    friends = df['user.friends_count']
    listed = df['user.listed_count']
    statuses = df['user.statuses_count']

    #ratio features
    df['ratio_followers_friends'] = followers / (friends + 1)
    df['ratio_listed_followers'] = listed / (followers + 1)
    df['reciprocity_score'] = (friends - followers) / (friends + followers + 1)
    #daily activity
    df['tweets_per_day'] = statuses / (df['account_age_days'] + 1)
    df['ratio_mention_status'] = df['num_mentions'] / (statuses + 1)
    total_engagement = df['retweet_count'] + df['favorite_count'] + df['quote_count'] + df['reply_count']
    df['total_tweet_engagement'] = total_engagement / (followers + 1)

    df['final_text'] = df.get('extended_tweet.full_text', df.get('text', pd.Series(''))).fillna('')
    df['final_text'] = df['final_text'].where(df['final_text'] != '', df.get('text', '')).fillna('')

    df['text_length'] = df['final_text'].astype(str).apply(len)
    df['bio_length'] = df.get('user.description', '').astype(str).apply(len)

    features_to_keep = [
        # user basic metrics
        'user.followers_count', 'user.friends_count', 'user.listed_count',
        'user.favourites_count', 'user.statuses_count',
        # tweet basic metrics
        'retweet_count', 'favorite_count', 'quote_count', 'reply_count',
        # ratios
        'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days',
        'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement',
        # account qualities
        'is_verified', 'is_default_profile', 'is_default_image', 'is_geo_enabled',
        'is_protected', 'has_url',
        # time related features
        'tweet_hour', 'tweet_is_weekend',
        # text features
        'text_length', 'bio_length',
        # entity counts
        'num_urls', 'num_hashtags', 'num_mentions', 'has_media',
    ]

    final_cols = [c for c in features_to_keep if c in df.columns]

    return df[final_cols].fillna(0)

In [ ]:
def create_nlp_features(df_train, df_test, y_train):
    """
    Crée des features NLP (TF-IDF des Bios et Sentiment des Tweets).

    Args:
        df_train (pd.DataFrame): DataFrame d'entraînement complet (X_train).
        df_test (pd.DataFrame): DataFrame de test (X_kaggle).
        y_train (pd.Series): Cible d'entraînement (y_train).

    Returns:
        tuple: (df_train_nlp, df_test_nlp) avec les nouvelles colonnes.
    """
    #french_stopwords = nltk.stopwords.words("french")
    train_bio = df_train.get('user.description', pd.Series([''] * len(df_train))).fillna('').astype(str)
    test_bio = df_test.get('user.description', pd.Series([''] * len(df_test))).fillna('').astype(str)


    def get_final_text(df):
        text = df.get('text', pd.Series([''] * len(df))).fillna('')
        full_text = df.get('extended_tweet.full_text', text).fillna(text)
        return full_text.astype(str)

    train_text = get_final_text(df_train)
    test_text = get_final_text(df_test)

    #TF-IDF

    print("TF-IDF vectorization du corps du tweet")

    def clean_text(text):
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        return text

    train_text_clean = train_text.apply(clean_text)
    test_text_clean = test_text.apply(clean_text)

    tfidf = TfidfVectorizer(max_features=1000, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf_tweet = tfidf.fit_transform(train_text_clean)
    X_test_tfidf_tweet = tfidf.transform(test_text_clean)

    log_reg = LogisticRegression(solver='sag', random_state=42)
    log_reg.fit(X_train_tfidf_tweet, y_train.astype(int))

    train_tweet_proba = log_reg.predict_proba(X_train_tfidf_tweet)[:, 1]
    test_tweet_proba = log_reg.predict_proba(X_test_tfidf_tweet)[:, 1]


    # additional meta feature : TF-IDF on bios

    print("  -> Calcul du TF-IDF sur les Bios et entraînement du Méta-Modèle...")

    train_bio_clean = train_bio.apply(clean_text)
    test_bio_clean = test_bio.apply(clean_text)

    tfidf = TfidfVectorizer(max_features=1000, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf = tfidf.fit_transform(train_bio_clean)
    X_test_tfidf = tfidf.transform(test_bio_clean)

    log_reg = LogisticRegression(solver='liblinear', random_state=42)
    log_reg.fit(X_train_tfidf, y_train.astype(int))

    train_bio_proba = log_reg.predict_proba(X_train_tfidf)[:, 1]
    test_bio_proba = log_reg.predict_proba(X_test_tfidf)[:, 1]

    # feeling analysis

    print("  -> Extraction du Sentiment (Polarity/Subjectivity) des Tweets...")

    # TextBlob version, which was not the best for French feeling analysis but it was the first one we thought of

    def get_sentiment(text):
        try:
            analysis = TextBlob(text)
            return pd.Series({'polarity': analysis.sentiment.polarity, 'subjectivity': analysis.sentiment.subjectivity})
        except:
            return pd.Series({'polarity': 0.0, 'subjectivity': 0.0})

    train_sentiment = train_text.apply(get_sentiment)
    test_sentiment = test_text.apply(get_sentiment)


    #NLP features

    df_train_nlp = pd.DataFrame({
        'meta_bio_proba': train_bio_proba,
        'tweet_polarity': train_sentiment['polarity'],
        'tweet_subjectivity': train_sentiment['subjectivity'],
        'meta_tweet_proba': train_tweet_proba
    })

    df_test_nlp = pd.DataFrame({
        'meta_bio_proba': test_bio_proba,
        'tweet_polarity': test_sentiment['polarity'],
        'tweet_subjectivity': test_sentiment['subjectivity'],
        'meta_tweet_proba': test_tweet_proba
    })

    return df_train_nlp, df_test_nlp

In [ ]:
# feature concatenation
print("Construction des features avancées (Métadonnées)...")

X_train_advanced = create_advanced_features(X_train)
X_kaggle_advanced = create_advanced_features(X_kaggle)

y_train_clean = y_train.astype(int)

X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"\nFeatures combinées ({len(X_train_advanced.columns)}):")
print(list(X_train_advanced.columns))

In [ ]:
########################################################################################
# basic MiniLM for adding content embeddings to the NLP features to give to the XGBoost
########################################################################################

model_name = "nreimers/MiniLM-L6-H384-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
model.to("cuda")

#In the next cell : fetching our own model after unzipping the lora_minilm_out.zip file

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

def safe_cast_to_str(series):
    return series.fillna("").astype(str).apply(lambda x: str(x))


def get_final_text(df):
    if 'extended_tweet.full_text' in df.columns:
        return safe_cast_to_str(df['extended_tweet.full_text'])
    elif 'text' in df.columns:
        return safe_cast_to_str(df['text'])
    else:
        return pd.Series([""] * len(df))


def get_miniLM_embeddings(texts, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = [str(t) for t in texts[i:i+batch_size]]   # FIXED here

        encoded_input = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        encoded_input = {k: v.to("cuda") for k, v in encoded_input.items()}

        with torch.no_grad():
            model_output = model(**encoded_input)

        embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings).numpy()

X_train_text = get_final_text(X_train)
X_kaggle_text = get_final_text(X_kaggle)

print("✓ Text fields extracted for MiniLM")

print("🔄 Generating MiniLM embeddings for train...")
X_train_vectors = get_miniLM_embeddings(X_train_text)

print("🔄 Generating MiniLM embeddings for Kaggle test...")
X_kaggle_vectors = get_miniLM_embeddings(X_kaggle_text)


EMBEDDING_DIM = X_train_vectors.shape[1]
embed_cols = [f'miniLM_e_{i}' for i in range(EMBEDDING_DIM)]

X_train_nlp = pd.DataFrame(X_train_vectors, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vectors, columns=embed_cols)

X_train_advanced = pd.concat(
    [X_train_advanced.reset_index(drop=True), X_train_nlp.reset_index(drop=True)],
    axis=1
)
X_kaggle_advanced = pd.concat(
    [X_kaggle_advanced.reset_index(drop=True), X_kaggle_nlp.reset_index(drop=True)],
    axis=1
)

print("\n" + "=" * 50)
print("✓ MiniLM embeddings fused with features")
print("=" * 50)
print(f"Total features in training set: {X_train_advanced.shape[1]}")
print(f"Total features in Kaggle set: {X_kaggle_advanced.shape[1]}")


params = {
    'n_estimators': [300],
    'learning_rate': [0.1],
    'max_depth': [6, 10, 15],
    'subsample': [0.9],
    'colsample_bytree': [0.8, 0.9],
    'gamma': [0.5]
}

xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist'
)

random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=params,
    n_iter=3,
    scoring='accuracy',
    cv=4,
    verbose=1,
    n_jobs=4,
    random_state=42
)

print("Starting Randomized Search with XGBoost...")
random_search.fit(X_train_advanced, y_train_clean)

print(f"Best CV Accuracy: {random_search.best_score_:.4f}")

best_model = random_search.best_estimator_
y_pred_kaggle = best_model.predict(X_kaggle_advanced)

output = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": y_pred_kaggle
})

output.to_csv('submission_xgboost_advanced_final.csv', index=False)

print("Submission file generated: submission_xgboost_advanced_final.csv")
print(output.head())

from google.colab import files
files.download('submission_xgboost_advanced_final.csv')

In [ ]:
################################################################
#Lightweight basic MiniLM embeddings + NLP features -> XGBoost
################################################################

from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
import xgboost as xgb

model_name = "sentence-transformers/all-MiniLM-L6-v2"
#384-dim : still small and much lighter than 768-dim models

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
model.to("cuda")

model.half()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).half()
    masked = token_embeddings * mask
    return masked.sum(1) / mask.sum(1).clamp(min=1e-9)

def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df))

def embed_text(text_series, batch_size=32):
    embeddings = np.zeros((len(text_series), 384), dtype=np.float16)

    for i in range(0, len(text_series), batch_size):
        batch = list(text_series.iloc[i:i+batch_size])
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64, #cutting necessary memory in HALF
        ).to("cuda")

        with torch.no_grad():
            out = model(**inputs)

        pooled = mean_pooling(out, inputs["attention_mask"])

        embeddings[i:i+batch_size] = pooled.float().cpu().numpy()

        torch.cuda.empty_cache()

    return embeddings

X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("Text extracted")


print("Embedding train...")
X_train_vec = embed_text(X_train_text)

print("Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)

embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]

X_train_nlp = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

X_train_final = pd.concat(
    [X_train_advanced.reset_index(drop=True), X_train_nlp], axis=1
)
X_kaggle_final = pd.concat(
    [X_kaggle_advanced.reset_index(drop=True), X_kaggle_nlp], axis=1
)

print("Embeddings merged")

xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(X_train_final, y_train_clean)

preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_lightweight.csv", index=False)

print("Saved submission_lightweight.csv")
submission.head()

from google.colab import files
files.download('submission_lightweight.csv')

In [ ]:
#######################################################
#using lora finetuned weights for the MiniLM embeddings
#######################################################

BASE_MODEL = "nreimers/MiniLM-L6-H384-uncased"
ADAPTER_PATH = "/content/content/minilm_lora_adapter"


print("Loading tokenizer and classification base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    torch_dtype=torch.float16
)

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.to("cuda")
model.eval()

print("Model loaded on", next(model.parameters()).device)

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.hidden_states[-1]   # <-- BETTER: use hidden states
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).to(token_embeddings.dtype)
    return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

model.config.output_hidden_states = True

def embed_text(text_series, batch_size=32):
    embeddings = np.zeros((len(text_series), 384), dtype=np.float16)

    for i in range(0, len(text_series), batch_size):
        batch = text_series.iloc[i:i+batch_size].tolist()

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                output_hidden_states=True
            )

        pooled = mean_pooling(out, enc["attention_mask"])

        embeddings[i:i+batch_size] = pooled.float().cpu().numpy()
        torch.cuda.empty_cache()

    return embeddings

X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("✓ Text extracted")

print("Embedding train...")
X_train_vec = embed_text(X_train_text)

print("Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)

embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]

X_train_nlp = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

X_train_final = pd.concat(
    [X_train_advanced.reset_index(drop=True), X_train_nlp], axis=1
)
X_kaggle_final = pd.concat(
    [X_kaggle_advanced.reset_index(drop=True), X_kaggle_nlp], axis=1
)
print("Embeddings merged")

xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)

preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_lightweight.csv", index=False)

print("Saved submission_lightweight.csv")
submission.head()

from google.colab import files
files.download('submission_lightweight.csv')

In [ ]:
#########################################################
#Trying the stronger embedding model (BGE-small-en-v1.5)
#########################################################

MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("🔍 Loading BGE-small-en-v1.5...")
model = SentenceTransformer(MODEL_NAME, device="cuda")
embed_dim = model.get_sentence_embedding_dimension()
print("✓ Model loaded on GPU — embedding dim:", embed_dim)


def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df))


def embed_text(text_series, batch_size=64):
    texts = text_series.tolist()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cuda"
    )

    return embeddings.astype(np.float16)


print("Extracting text...")
X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("Embedding train...")
X_train_vec = embed_text(X_train_text)

print("Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)


embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]

X_train_nlp = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

X_train_final = pd.concat([X_train_advanced.reset_index(drop=True),
                           X_train_nlp], axis=1)

X_kaggle_final = pd.concat([X_kaggle_advanced.reset_index(drop=True),
                            X_kaggle_nlp], axis=1)

print("Embeddings merged with advanced features")

xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)

preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_bge_small.csv", index=False)

print("Saved submission_bge_small.csv")
submission.head()

from google.colab import files
files.download('submission_bge_small.csv')

In [ ]:
########################################################################
#trying a vinai/bertweet-base model for the embeddings instead of MiniLM
########################################################################
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd

MODEL_NAME = "vinai/bertweet-base"

print("🔍 Loading BERTweet...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModel.from_pretrained(MODEL_NAME).to("cuda")
model.eval()

print("✓ BERTweet ready")

def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df))


def embed_text(text_series, batch_size=32):
    texts = text_series.tolist()
    all_emb = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)
            # Mean pooling
            emb = outputs.last_hidden_state.mean(dim=1)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)

        all_emb.append(emb.cpu().numpy())

        torch.cuda.empty_cache()

    return np.concatenate(all_emb, axis=0).astype(np.float16)

print("Construction des features avancées (Métadonnées)...")

# Provided by your earlier code
X_train_advanced = create_advanced_features(X_train)
X_kaggle_advanced = create_advanced_features(X_kaggle)

y_train_clean = y_train.astype(int)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

X_meta_train, X_meta_val, y_meta_train, y_meta_val = train_test_split(
    X_train_advanced, y_train_clean, test_size=0.2, random_state=42, stratify=y_train_clean
)

# Provided by your earlier code
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

# Merge advanced + NLP features
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"\nFeatures combinées ({len(X_train_advanced.columns)}):")
print(list(X_train_advanced.columns))

print("🔄 Extracting text...")
X_train_text = get_text(X_train)
X_kaggle_text = get_text(X_kaggle)

print("🔄 Embedding train...")
X_train_vec = embed_text(X_train_text)

print("🔄 Embedding Kaggle...")
X_kaggle_vec = embed_text(X_kaggle_text)

# Convert to DataFrame
embed_cols = [f"e{i}" for i in range(X_train_vec.shape[1])]
X_train_embed = pd.DataFrame(X_train_vec, columns=embed_cols)
X_kaggle_embed = pd.DataFrame(X_kaggle_vec, columns=embed_cols)

# Merge all features
X_train_final = pd.concat([X_train_advanced.reset_index(drop=True),
                           X_train_embed], axis=1)

X_kaggle_final = pd.concat([X_kaggle_advanced.reset_index(drop=True),
                            X_kaggle_embed], axis=1)

print("✓ All embeddings merged with advanced + NLP features")

import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

print("🔄 Training XGBoost...")
xgb_model.fit(X_train_final, y_train_clean)

preds = xgb_model.predict(X_kaggle_final)

submission = pd.DataFrame({
    "ID": kaggle_data["challenge_id"],
    "Prediction": preds
})

submission.to_csv("submission_tweetransformer_small.csv", index=False)

print("\n📄 Saved submission_tweetransformer_small.csv")
submission.head()

from google.colab import files
files.download('submission_tweetransformer_small.csv')

## ALTERNATIVE PROGRAMS

In [ ]:
import os
import json
import math
import random
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from pandas import json_normalize
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
import nltk
from textblob import TextBlob
import re

# ------------------------------
# 0. Settings
# ------------------------------
MODEL_NAME = "jinaai/jina-embeddings-v3"
EMB_DTYPE = np.float16              # on-disk storage for raw embeddings
PCA_DIM = 256                       # compressed embedding dim
EMB_BATCH = 64                      # embedding batch size (reduce if OOM)
PCA_BATCH = 1024                    # rows per incremental PCA partial_fit
TFIDF_TWEET_MAXFEAT = 2000         # reduce for memory
TFIDF_BIO_MAXFEAT = 2000
MEMMAP_DIR = "/content/emb_memmaps"
os.makedirs(MEMMAP_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ------------------------------
# 1. Robust JSONL loader
# ------------------------------
def load_jsonl_skip_bad(path):
    data_list = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                data_list.append(json.loads(line))
            except json.JSONDecodeError:
                # skip malformed line silently
                continue
    return json_normalize(data_list)

print("Loading data...")
train_df = load_jsonl_skip_bad("train.jsonl")
kaggle_df = load_jsonl_skip_bad("kaggle_test.jsonl")
print("Loaded rows:", len(train_df), "train /", len(kaggle_df), "kaggle")

# Ensure label exists
if "label" not in train_df.columns:
    raise RuntimeError("train.jsonl must contain 'label' column")

# create X/y
X_train_raw = train_df.copy()
y_train = X_train_raw["label"].astype(int)
X_kaggle_raw = kaggle_df.copy()

# ------------------------------
# 2. Feature engineering: advanced metadata (your function, slightly hardened)
# ------------------------------
def create_advanced_features(df_input):
    df = df_input.copy()
    # fallback series
    default_int_series = pd.Series(0, index=df.index)
    default_bool_series = pd.Series(False, index=df.index)

    # numeric columns with safe get
    df["user.followers_count"] = df.get("user.followers_count", default_int_series).fillna(0).astype(float)
    df["user.friends_count"] = df.get("user.friends_count", default_int_series).fillna(0).astype(float)
    df["user.listed_count"] = df.get("user.listed_count", default_int_series).fillna(0).astype(float)
    df["user.favourites_count"] = df.get("user.favourites_count", default_int_series).fillna(0).astype(float)
    df["user.statuses_count"] = df.get("user.statuses_count", default_int_series).fillna(0).astype(float)
    df["retweet_count"] = df.get("retweet_count", default_int_series).fillna(0).astype(float)
    df["favorite_count"] = df.get("favorite_count", default_int_series).fillna(0).astype(float)
    df["quote_count"] = df.get("quote_count", default_int_series).fillna(0).astype(float)
    df["reply_count"] = df.get("reply_count", default_int_series).fillna(0).astype(float)

    # dates
    df["user_created_at_dt"] = pd.to_datetime(df.get("user.created_at", None), errors="coerce")
    ref_date = pd.to_datetime("now", utc=True)
    df["account_age_days"] = (ref_date - df["user_created_at_dt"]).dt.days.fillna(0).astype(float)

    df["created_at_dt"] = pd.to_datetime(df.get("created_at", None), errors="coerce")
    df["tweet_hour"] = df["created_at_dt"].dt.hour.fillna(-1).astype(int)
    df["tweet_is_weekend"] = df["created_at_dt"].dt.dayofweek.isin([5,6]).fillna(False).astype(int)

    # booleans
    df["is_default_profile"] = df.get("user.default_profile", default_bool_series).fillna(False).astype(int)
    df["is_default_image"] = df.get("user.default_profile_image", default_bool_series).fillna(False).astype(int)
    df["is_verified"] = df.get("user.verified", default_bool_series).fillna(False).astype(int)
    df["is_protected"] = df.get("user.protected", default_bool_series).fillna(False).astype(int)
    df["has_url"] = df.get("user.url", pd.Series(None, index=df.index)).notna().astype(int)

    # entities counting
    def count_entities(x):
        if isinstance(x, list):
            return len(x)
        return 0
    df["num_urls"] = df.get("entities.urls", default_int_series).apply(lambda x: count_entities(x)).astype(float)
    df["num_hashtags"] = df.get("entities.hashtags", default_int_series).apply(lambda x: count_entities(x)).astype(float)
    df["num_mentions"] = df.get("entities.user_mentions", default_int_series).apply(lambda x: count_entities(x)).astype(float)
    df["has_media"] = df.get("extended_entities.media", default_bool_series).notna().astype(int)

    # ratios
    followers = df["user.followers_count"].replace(0, 0.0)
    friends = df["user.friends_count"].replace(0, 0.0)
    listed = df["user.listed_count"].replace(0, 0.0)
    statuses = df["user.statuses_count"].replace(0, 0.0)

    df["ratio_followers_friends"] = followers / (friends + 1.0)
    df["ratio_listed_followers"] = listed / (followers + 1.0)
    df["reciprocity_score"] = (friends - followers) / (friends + followers + 1.0)
    df["tweets_per_day"] = statuses / (df["account_age_days"] + 1.0)
    df["ratio_mention_status"] = df["num_mentions"] / (statuses + 1.0)
    total_engagement = df["retweet_count"] + df["favorite_count"] + df["quote_count"] + df["reply_count"]
    df["total_tweet_engagement"] = total_engagement / (followers + 1.0)

    # text lengths and final_text
    df["final_text"] = df.get("extended_tweet.full_text", df.get("text", "")).fillna("")
    df["final_text"] = df["final_text"].where(df["final_text"] != "", df.get("text", "")).fillna("")
    df["text_length"] = df["final_text"].astype(str).apply(len).astype(float)
    df["bio_length"] = df.get("user.description", "").fillna("").astype(str).apply(len).astype(float)

    features_to_keep = [
        'user.followers_count','user.friends_count','user.listed_count','user.favourites_count','user.statuses_count',
        'retweet_count','favorite_count','quote_count','reply_count',
        'ratio_followers_friends','ratio_listed_followers','tweets_per_day','account_age_days',
        'reciprocity_score','ratio_mention_status','total_tweet_engagement',
        'is_verified','is_default_profile','is_default_image','is_protected','has_url',
        'tweet_hour','tweet_is_weekend','text_length','bio_length',
        'num_urls','num_hashtags','num_mentions','has_media'
    ]

    final_cols = [c for c in features_to_keep if c in df.columns]
    return df[final_cols].fillna(0)

print("Building advanced features...")
X_train_meta = create_advanced_features(X_train_raw)
X_kaggle_meta = create_advanced_features(X_kaggle_raw)
print("Meta shapes:", X_train_meta.shape, X_kaggle_meta.shape)

# ------------------------------
# 3. NLP meta-features: TF-IDF + meta logistic models + TextBlob sentiment
# ------------------------------
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
french_stopwords = stopwords.words("french")

def get_final_text_series(df):
    text = df.get("extended_tweet.full_text", df.get("text", pd.Series([""] * len(df))))
    return text.fillna("").astype(str)

print("Preparing TF-IDF meta-features...")
train_texts_for_tfidf = get_final_text_series(X_train_raw)
kaggle_texts_for_tfidf = get_final_text_series(X_kaggle_raw)

# tweet TF-IDF (char_wb as you had)
tfidf_tweet = TfidfVectorizer(max_features=TFIDF_TWEET_MAXFEAT, stop_words=french_stopwords, ngram_range=(2,5), analyzer='char_wb', lowercase=False)
X_tfidf_train_tweet = tfidf_tweet.fit_transform(train_texts_for_tfidf)
X_tfidf_kaggle_tweet = tfidf_tweet.transform(kaggle_texts_for_tfidf)

# train meta logistic on tweet tfidf
logreg_tweet = LogisticRegression(solver="sag", max_iter=1000)
logreg_tweet.fit(X_tfidf_train_tweet, y_train)
train_tweet_proba = logreg_tweet.predict_proba(X_tfidf_train_tweet)[:,1]
kaggle_tweet_proba = logreg_tweet.predict_proba(X_tfidf_kaggle_tweet)[:,1]

# bio TF-IDF
train_bio = X_train_raw.get("user.description", pd.Series([""] * len(X_train_raw))).fillna("").astype(str)
kaggle_bio = X_kaggle_raw.get("user.description", pd.Series([""] * len(X_kaggle_raw))).fillna("").astype(str)

tfidf_bio = TfidfVectorizer(max_features=TFIDF_BIO_MAXFEAT, stop_words=french_stopwords, ngram_range=(2,5), analyzer='char_wb', lowercase=False)
X_tfidf_train_bio = tfidf_bio.fit_transform(train_bio)
X_tfidf_kaggle_bio = tfidf_bio.transform(kaggle_bio)

logreg_bio = LogisticRegression(solver="liblinear", max_iter=1000)
logreg_bio.fit(X_tfidf_train_bio, y_train)
train_bio_proba = logreg_bio.predict_proba(X_tfidf_train_bio)[:,1]
kaggle_bio_proba = logreg_bio.predict_proba(X_tfidf_kaggle_bio)[:,1]

# sentiment using TextBlob (note: English-oriented, but works reasonably)
def get_sentiment_series(series):
    out_polarity = []
    out_subjectivity = []
    for txt in tqdm(series, desc="Sentiment", leave=False):
        try:
            tb = TextBlob(str(txt))
            out_polarity.append(tb.sentiment.polarity)
            out_subjectivity.append(tb.sentiment.subjectivity)
        except:
            out_polarity.append(0.0)
            out_subjectivity.append(0.0)
    return np.array(out_polarity), np.array(out_subjectivity)

train_tweet_polarity, train_tweet_subjectivity = get_sentiment_series(train_texts_for_tfidf)
kaggle_tweet_polarity, kaggle_tweet_subjectivity = get_sentiment_series(kaggle_texts_for_tfidf)

# Collect NLP meta-features into DataFrames
X_train_nlp_meta = pd.DataFrame({
    "meta_bio_proba": train_bio_proba,
    "meta_tweet_proba": train_tweet_proba,
    "tweet_polarity": train_tweet_polarity,
    "tweet_subjectivity": train_tweet_subjectivity
})

X_kaggle_nlp_meta = pd.DataFrame({
    "meta_bio_proba": kaggle_bio_proba,
    "meta_tweet_proba": kaggle_tweet_proba,
    "tweet_polarity": kaggle_tweet_polarity,
    "tweet_subjectivity": kaggle_tweet_subjectivity
})

print("NLP meta features shapes:", X_train_nlp_meta.shape, X_kaggle_nlp_meta.shape)

# Merge metadata + nlp meta
X_train_meta = pd.concat([X_train_meta.reset_index(drop=True), X_train_nlp_meta.reset_index(drop=True)], axis=1)
X_kaggle_meta = pd.concat([X_kaggle_meta.reset_index(drop=True), X_kaggle_nlp_meta.reset_index(drop=True)], axis=1)

# ------------------------------
# 4. Prepare text lists for embeddings (final_text)
# ------------------------------
def get_final_text_list(df):
    if "extended_tweet.full_text" in df.columns:
        s = df["extended_tweet.full_text"].fillna("").astype(str)
    else:
        s = df.get("text", pd.Series([""] * len(df))).fillna("").astype(str)
    return s.tolist()

train_texts = get_final_text_list(X_train_raw)
kaggle_texts = get_final_text_list(X_kaggle_raw)
print("Texts ready:", len(train_texts), len(kaggle_texts))

# ------------------------------
# 5. Load Jina model (encoder) for embeddings
# ------------------------------
print("Loading embedding model:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
torch_dtype = torch.float16 if DEVICE == "cuda" else None
embedder = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype)
embedder = embedder.to(DEVICE)
embedder.eval()
if DEVICE == "cuda":
    try:
        embedder.half()
    except Exception:
        pass

EMB_SIZE = embedder.config.hidden_size
print("Embedder hidden size:", EMB_SIZE)

# ------------------------------
# 6. Stream embeddings to memmap (train + kaggle)
# ------------------------------
def make_memmap(path, shape, dtype=np.float16):
    return np.memmap(path, dtype=dtype, mode="w+", shape=shape)

train_emb_path = os.path.join(MEMMAP_DIR, "train_emb.dat")
kaggle_emb_path = os.path.join(MEMMAP_DIR, "kaggle_emb.dat")

train_mm = make_memmap(train_emb_path, (len(train_texts), EMB_SIZE), dtype=EMB_DTYPE)
kaggle_mm = make_memmap(kaggle_emb_path, (len(kaggle_texts), EMB_SIZE), dtype=EMB_DTYPE)

def mean_pooling(model_output, attention_mask):
    token_emb = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_emb.size()).to(token_emb.dtype)
    summed = (token_emb * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return (summed / denom)

def stream_embed(texts, memmap_arr, batch_size=EMB_BATCH):
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        end = min(start + batch_size, len(texts))
        batch = texts[start:end]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
        encoded = {k:v.to(DEVICE) for k,v in encoded.items()}
        with torch.no_grad():
            out = embedder(**encoded)
            pooled = mean_pooling(out, encoded["attention_mask"]).to("cpu").numpy()
        # normalize
        norms = np.linalg.norm(pooled, axis=1, keepdims=True).clip(min=1e-9)
        pooled = pooled / norms
        memmap_arr[start:end, :] = pooled.astype(EMB_DTYPE)
        torch.cuda.empty_cache()
    memmap_arr.flush()

print("Streaming train embeddings...")
stream_embed(train_texts, train_mm, batch_size=EMB_BATCH)
print("Streaming kaggle embeddings...")
stream_embed(kaggle_texts, kaggle_mm, batch_size=EMB_BATCH)

# reopen as read-only memmaps
train_embs = np.memmap(train_emb_path, dtype=EMB_DTYPE, mode="r", shape=(len(train_texts), EMB_SIZE))
kaggle_embs = np.memmap(kaggle_emb_path, dtype=EMB_DTYPE, mode="r", shape=(len(kaggle_texts), EMB_SIZE))

# ------------------------------
# 7. Incremental PCA to compress embeddings -> PCA_DIM
# ------------------------------
print("Running IncrementalPCA to", PCA_DIM, "dims")
ipca = IncrementalPCA(n_components=PCA_DIM, batch_size=PCA_BATCH)

for start in tqdm(range(0, train_embs.shape[0], PCA_BATCH), desc="IPCA fit"):
    end = min(start + PCA_BATCH, train_embs.shape[0])
    chunk = np.asarray(train_embs[start:end]).astype(np.float32)
    ipca.partial_fit(chunk)

def ipca_transform_to_memmap(src_memmap, out_path, n_components=PCA_DIM, batch_size=PCA_BATCH):
    out_mm = np.memmap(out_path, dtype=np.float32, mode="w+", shape=(src_memmap.shape[0], n_components))
    for start in tqdm(range(0, src_memmap.shape[0], batch_size), desc=f"IPCA transform {os.path.basename(out_path)}"):
        end = min(start + batch_size, src_memmap.shape[0])
        chunk = np.asarray(src_memmap[start:end]).astype(np.float32)
        out_mm[start:end] = ipca.transform(chunk)
    out_mm.flush()
    return out_mm

train_pca_path = os.path.join(MEMMAP_DIR, "train_emb_pca.npy")
kaggle_pca_path = os.path.join(MEMMAP_DIR, "kaggle_emb_pca.npy")
train_pca_mm = ipca_transform_to_memmap(train_embs, train_pca_path, n_components=PCA_DIM)
kaggle_pca_mm = ipca_transform_to_memmap(kaggle_embs, kaggle_pca_path, n_components=PCA_DIM)

train_pca = np.memmap(train_pca_path, dtype=np.float32, mode="r", shape=(len(train_texts), PCA_DIM))
kaggle_pca = np.memmap(kaggle_pca_path, dtype=np.float32, mode="r", shape=(len(kaggle_texts), PCA_DIM))

print("PCA shapes:", train_pca.shape, kaggle_pca.shape)

# ------------------------------
# 8. Combine sparse TF-IDF (tweet + bio) + meta features + PCA embeddings
# We'll convert PCA embeddings to dense csr to hstack with sparse TFIDF.
# ------------------------------
print("Combining features...")

# Build sparse TFIDF for tweets (we already have tweet tfidf matrices)
# Combine tweet tfidf and bio tfidf? We used both as meta logit models, but we can also include raw TFIDF.
# For memory reasons, we'll include the tweet TF-IDF sparse matrix only (X_tfidf_train_tweet).
# If you want, change to include bio tfidf as well.

# Create CSR for PCA embeddings
train_pca_csr = csr_matrix(train_pca)
kaggle_pca_csr = csr_matrix(kaggle_pca)

# Build final sparse matrices: [tweet_tfidf | pca_emb | sentiment/meta small dense]
# First build sentiment/meta sparse matrix
meta_train_dense = X_train_meta.values.astype(np.float32)
meta_kaggle_dense = X_kaggle_meta.values.astype(np.float32)

# Convert dense meta to csr
meta_train_csr = csr_matrix(meta_train_dense)
meta_kaggle_csr = csr_matrix(meta_kaggle_dense)

# Also include the meta logistic probs & polarity/subjectivity we computed earlier are already in meta.

X_train_sparse = hstack([X_tfidf_train_tweet, train_pca_csr, meta_train_csr], format="csr")
X_kaggle_sparse = hstack([X_tfidf_kaggle_tweet, kaggle_pca_csr, meta_kaggle_csr], format="csr")

print("Final sparse shapes:", X_train_sparse.shape, X_kaggle_sparse.shape)

# ------------------------------
# 9. Train/validation split (use small val for early stopping)
# ------------------------------
print("Splitting train/val...")
X_tr, X_val, y_tr, y_val = train_test_split(X_train_sparse, y_train.values, test_size=0.12, random_state=42, stratify=y_train)

# ------------------------------
# 10. LightGBM dataset & training with early stopping
# ------------------------------
print("Training LightGBM (with early stopping)...")
lgb_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": 8,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": 42,
    "n_jobs": 4
}

dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

bst = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dval],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=50)]
)

# ------------------------------
# 11. Predict on Kaggle
# ------------------------------
print("Predicting Kaggle set...")
preds_prob = bst.predict(X_kaggle_sparse, num_iteration=bst.best_iteration)
preds_label = (preds_prob >= 0.5).astype(int)

submission = pd.DataFrame({
    "ID": X_kaggle_raw.get("challenge_id", X_kaggle_raw.index).reset_index(drop=True),
    "Prediction": preds_label
})

out_path = "submission_jina_v3_lightgbm.csv"
submission.to_csv(out_path, index=False)
print("Saved submission:", out_path, "shape:", submission.shape)

# optional download (Colab)
try:
    from google.colab import files
    files.download(out_path)
except Exception:
    pass

print("Pipeline complete.")

In [ ]:
##code to use with any classifier to use the user profile
USER_ID_COLUMN = 'user.profile_banner_url'

y_pred_proba_kaggle = random_search.predict(X_kaggle_advanced)
y_pred_kaggle = (y_pred_proba_kaggle > 0.5).astype(int)

df_kaggle_preds = pd.DataFrame({
    'challenge_id': X_kaggle['challenge_id'],
    'user_id_key': X_kaggle[USER_ID_COLUMN], # Clé d'utilisateur
    'y_pred_tweet': y_pred_kaggle            # Prédiction individuelle (le secours)
})


user_pred_mean = df_kaggle_preds.groupby('user_id_key')['y_pred_tweet'].mean()
user_majority_vote = np.where(user_pred_mean >= 0.5, 1, 0)

# Convertir la Série des votes de majorité en DataFrame pour la fusion
df_majority_vote = pd.DataFrame({
    'user_id_key': user_pred_mean.index,
    'y_pred_user_majority': user_majority_vote
})

df_final_preds = pd.merge(
    df_kaggle_preds,
    df_majority_vote,
    on='user_id_key',
    how='left'
)

df_final_preds['y_pred_final'] = df_final_preds['y_pred_user_majority'].fillna(
    df_final_preds['y_pred_tweet']
)
y_final_submission = df_final_preds['y_pred_final']
output = pd.DataFrame({
    'ID': df_final_preds['challenge_id'],
    "Prediction": y_final_submission
})

output['Prediction'] = output['Prediction'].astype(int)
output.to_csv('submission_lightgbm_majority_vote_final_nan_fixed.csv', index=False)

In [1]:
##bertweet
!pip install transformers sentencepiece accelerate bitsandbytes --quiet
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet
!pip install textblob fr-core-news-sm spacy --quiet
!python -m spacy download fr_core_news_sm

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
import spacy
import numpy as np
from scipy.sparse import hstack, csr_matrix

# -------------------------------------------------------
# 1) Load French NLP for lemmatization
# -------------------------------------------------------
nlp = spacy.load("fr_core_news_sm")

def preprocess_text(txt):
    """Clean + Lemmatize French text."""
    if not isinstance(txt, str):
        return ""
    doc = nlp(txt.lower())
    return " ".join([t.lemma_ for t in doc if not t.is_stop])

# -------------------------------------------------------
# 2) Load powerful multilingual embedding model
# -------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "BAAI/bge-m3"   # ⭐ Very strong multilingual embeddings (incl. French)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# -------------------------------------------------------
# 3) Embedding function
# -------------------------------------------------------
def embed_sentences(texts, batch_size=16):
    """
    Returns a dense embedding matrix (numpy).
    Uses mean pooling over the last hidden state.
    """
    all_embs = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)
            last_hidden = outputs.last_hidden_state
            attention_mask = encoded["attention_mask"]

            mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
            sum_embeddings = torch.sum(last_hidden * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            emb = (sum_embeddings / sum_mask).cpu().numpy()

        all_embs.append(emb)

    return np.vstack(all_embs)

# -------------------------------------------------------
# 4) Sentiment Analysis (English-based but works for polarity detection)
# -------------------------------------------------------
def sentiment_score(text):
    if not isinstance(text, str):
        return 0
    try:
        return TextBlob(text).sentiment.polarity
    except:
        return 0

# -------------------------------------------------------
# 5) LOAD YOUR DATA (replace filenames)
# -------------------------------------------------------
train = pd.read_csv("train_clean.csv")   # must contain: "text" + "label"
test = pd.read_csv("test_clean.csv")     # must contain: "text"

# -------------------------------------------------------
# 6) Preprocess text
# -------------------------------------------------------
train["clean"] = train["text"].apply(preprocess_text)
test["clean"]  = test["text"].apply(preprocess_text)

# -------------------------------------------------------
# 7) TF-IDF features
# -------------------------------------------------------
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_tfidf_train = tfidf.fit_transform(train["clean"])
X_tfidf_test  = tfidf.transform(test["clean"])

# -------------------------------------------------------
# 8) Sentiment + length
# -------------------------------------------------------
train["sentiment"] = train["text"].apply(sentiment_score)
test["sentiment"]  = test["text"].apply(sentiment_score)

train["length"] = train["text"].apply(lambda x: len(str(x)))
test["length"]  = test["text"].apply(lambda x: len(str(x)))

X_sentiment_train = csr_matrix(train[["sentiment", "length"]].values)
X_sentiment_test  = csr_matrix(test[["sentiment", "length"]].values)

# -------------------------------------------------------
# 9) Embeddings (the powerful part)
# -------------------------------------------------------
print("Embedding TRAIN...")
X_embed_train = embed_sentences(train["text"].tolist())

print("Embedding TEST...")
X_embed_test = embed_sentences(test["text"].tolist())

# Convert embeddings to sparse CSR for concatenation
X_embed_train_sparse = csr_matrix(X_embed_train)
X_embed_test_sparse  = csr_matrix(X_embed_test)

# -------------------------------------------------------
# 10) CONCAT ALL FEATURES
# -------------------------------------------------------
from scipy.sparse import hstack

X_train_full = hstack([X_tfidf_train, X_embed_train_sparse, X_sentiment_train])
X_test_full  = hstack([X_tfidf_test,  X_embed_test_sparse,  X_sentiment_test])

y_train = train["label"]

print("Final shapes:")
print("Train:", X_train_full.shape)
print("Test:",  X_test_full.shape)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 66.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'train_clean.csv'

In [ ]:
#with accuracy and loss for the three stages : only raw data, raw+ engineered features, all the above + content camembert embeddings

########################################################################
# ✅ FULL MULTI-STAGE PIPELINE WITH METRICS + FRENCH EMBEDDINGS (SAFE)
########################################################################

!pip install -q lightgbm sentence-transformers

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
from transformers import AutoTokenizer, AutoModel
import xgboost as xgb
import gc

########################################################################
# ✅ 1) TARGET + SPLIT
########################################################################

y_train_clean = y_train.astype(int)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train_clean, test_size=0.2, random_state=42, stratify=y_train_clean
)

########################################################################
# ✅ 2) STAGE 1 — RAW FEATURES ONLY
########################################################################

print("\n==================== STAGE 1: RAW FEATURES ONLY ====================")

raw_cols = X_train_split.select_dtypes(include=[np.number]).columns

Xtr_raw = X_train_split[raw_cols]
Xval_raw = X_val_split[raw_cols]

xgb_raw = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42
)

xgb_raw.fit(
    Xtr_raw, y_train_split,
    eval_set=[(Xval_raw, y_val_split)],
    verbose=False
)

raw_preds = xgb_raw.predict(Xval_raw)
raw_proba = xgb_raw.predict_proba(Xval_raw)

raw_acc = accuracy_score(y_val_split, raw_preds)
raw_loss = log_loss(y_val_split, raw_proba)

print(f"✅ RAW Accuracy: {raw_acc:.4f}")
print(f"✅ RAW LogLoss:  {raw_loss:.4f}")

########################################################################
# ✅ 3) STAGE 2 — RAW + ENGINEERED NLP FEATURES
########################################################################

print("\n==================== STAGE 2: RAW + ENGINEERED ====================")

X_train_adv = create_advanced_features(X_train)
X_kaggle_adv = create_advanced_features(X_kaggle)

X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)

X_train_full = pd.concat([X_train_adv, X_train_nlp], axis=1)

Xtr_full = X_train_full.iloc[X_train_split.index]
Xval_full = X_train_full.iloc[X_val_split.index]

xgb_features = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42
)

xgb_features.fit(
    Xtr_full, y_train_split,
    eval_set=[(Xval_full, y_val_split)],
    verbose=False
)

feat_preds = xgb_features.predict(Xval_full)
feat_proba = xgb_features.predict_proba(Xval_full)

feat_acc = accuracy_score(y_val_split, feat_preds)
feat_loss = log_loss(y_val_split, feat_proba)

print(f"✅ FEATURES Accuracy: {feat_acc:.4f}")
print(f"✅ FEATURES LogLoss:  {feat_loss:.4f}")

########################################################################
# ✅ 4) STAGE 3 — FRENCH CAMEMBERT EMBEDDINGS + ALL FEATURES
########################################################################

print("\n==================== STAGE 3: + FRENCH EMBEDDINGS ====================")

MODEL_NAME = "dangvantuan/sentence-camembert-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to("cuda")
model.eval()

def get_text(df):
    if "extended_tweet.full_text" in df.columns:
        return df["extended_tweet.full_text"].fillna("").astype(str)
    if "text" in df.columns:
        return df["text"].fillna("").astype(str)
    return pd.Series([""] * len(df))

def embed_text(text_series, batch_size=16):
    texts = text_series.tolist()
    all_emb = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        tok = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            out = model(**tok)
            emb = out.last_hidden_state.mean(dim=1)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)

        all_emb.append(emb.cpu().numpy())
        torch.cuda.empty_cache()

    return np.vstack(all_emb).astype(np.float16)

X_text = get_text(X_train)

X_embed = embed_text(X_text)

X_embed_df = pd.DataFrame(X_embed, columns=[f"e{i}" for i in range(X_embed.shape[1])])

X_train_all = pd.concat([X_train_full.reset_index(drop=True),
                          X_embed_df.reset_index(drop=True)], axis=1)

Xtr_all = X_train_all.iloc[X_train_split.index]
Xval_all = X_train_all.iloc[X_val_split.index]

xgb_all = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42
)

xgb_all.fit(
    Xtr_all, y_train_split,
    eval_set=[(Xval_all, y_val_split)],
    verbose=False
)

all_preds = xgb_all.predict(Xval_all)
all_proba = xgb_all.predict_proba(Xval_all)

all_acc = accuracy_score(y_val_split, all_preds)
all_loss = log_loss(y_val_split, all_proba)

print(f"✅ ALL FEATURES Accuracy: {all_acc:.4f}")
print(f"✅ ALL FEATURES LogLoss:  {all_loss:.4f}")

########################################################################
# ✅ 5) COMPARISON PLOT
########################################################################

labels = ["RAW", "RAW+NLP", "FULL+EMB"]
accs = [raw_acc, feat_acc, all_acc]
losses = [raw_loss, feat_loss, all_loss]

plt.figure()
plt.plot(labels, accs, marker="o")
plt.title("Accuracy by Feature Set")
plt.show()

plt.figure()
plt.plot(labels, losses, marker="o")
plt.title("LogLoss by Feature Set")
plt.show()

########################################################################
# ✅ FINAL VERDICT
########################################################################

print("\n==================== FINAL SCORE COMPARISON ====================")
for l,a,o in zip(labels, accs, losses):
    print(f"{l:12} | Accuracy = {a:.4f} | LogLoss = {o:.4f}")
